## Workshop Lab 7: Interactive Map Dashboards in Notebooks

In this notebook you will take a cleaned dataset (e.g. cleaned_coyotes_data.csv) and build out an interactive HTML map two different ways. First, with folium (which can be installed into any environment). Second, with argis map widgets, which require ESRI to use.

You can then share these HTML maps directly with colleagues by email!

### Step 1: Load a cleaned data set from notebook 6.

In [ ]:
import pandas as pd
import os

# Define path to working data folder
working_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "working"))

# Choose which batch species CSV you want to map:
# Options: "batch_ponderosa_pine.csv", "batch_prickly_pear.csv", "batch_yucca.csv"
file_name = "csv_name" 
csv_path = os.path.join(working_dir, file_name)

df = pd.read_csv(csv_path)
print(f"Successfully loaded {len(df)} records for interactive mapping.")
df.head(3)

### Method A: An Interactive Map with Folium

We used `Folium` in notebook 2, so you should have it installed in your env. `Folium` is built on Leaflet.js and helps with making lightweight maps that run in any web browser. You do not need an ArcGIS license to view these maps.

In [ ]:
import pandas as pd
import folium
import base64
from IPython.display import IFrame, display

# Load data
df = pd.read_csv("../04_data/working/batch_ponderosa_pine.csv")

# Create map (Esri base map tiles instead of OpenStreetMap)
m = folium.Map(
    location=[
        df['latitude'].mean(),
        df['longitude'].mean()
    ],
    zoom_start=11,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri"
)

# Add points
for _, row in df.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']]
    ).add_to(m)

# Render map in notebook
html = m.get_root().render()
encoded = base64.b64encode(html.encode("utf-8")).decode("utf-8")

display(IFrame(
    src=f"data:text/html;base64,{encoded}",
    width="100%",
    height=600
))

Now that we've seen our map, let's export it as an html file!

In [ ]:
# Save the map as a standalone HTML web page
output_html = os.path.abspath(os.path.join(os.getcwd(), "..", "04_data", "working", "interactive_wildlife_map.html"))
m.save(output_html)

print(f"Success! Standalone map saved to:\n{output_html}")
print("You can email this .html file to anyone—they can open it in Chrome or Edge without ArcGIS Pro!")

### Method B: Using ArcGIS Map Widgets
If you want to leverage Esri's web mapping infrastructure directly inside Jupyter using your portal credentials, you can use the ArcGIS API for Python map widget.

**NOTE** To run this code successfully you will need to install [widgets](https://github.com/microsoft/vscode-jupyter/wiki/IPyWidget-Support-in-VS-Code-Python). This will require you to have admin privileges for your computer and might not work on the student logins. Here is one method for doing this in VS Code:
1) At the very top of VS Code, click File, then Preferences, then Settings.
2) A Settings page opens with a search box at the top. Click in the box and type `widget script`.
3) Find the entry called Jupyter: Widget Script Sources. Under it, click the Add Item button (or edit JSON).
4) A box appears. Type `jsdelivr.com` and press Enter (or click OK).
5) Click Add Item again, type `unpkg.com` and press Enter.
6) Close the Settings tab. Click Save if prompted.
7) Go back to the notebook. In the toolbar at the top of the notebook, click Restart. This makes the notebook pick up the new setting.
8) Run the cells in this notebook from Method A again.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import GeoAccessor

# Connect to your active ArcGIS Pro session
gis = GIS("pro")
print(f"Connected to portal as: {gis.users.me.username}")

# Center the map on the same data as the Folium map
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

# Turn the table into points: each row gets a location from its longitude and latitude
sdf = GeoAccessor.from_xy(df, x_column='longitude', y_column='latitude', sr=4326)

# Create an ArcGIS interactive map widget, then set its center and zoom
m_arcgis = gis.map()
m_arcgis.center = [center_lat, center_lon]
m_arcgis.zoom = 7

# Draw the points on the map
sdf.spatial.plot(map_widget=m_arcgis)

# Display the ArcGIS map widget inline
print("ArcGIS Web Map Widget rendered below:")
m_arcgis

#### Adding Hosted or Local Layers to the ArcGIS Widget
If you published a hosted feature layer in Notebook 5 (or if you have an existing item ID in your AGOL content library), you can drop it straight into your widget view with a single line of code:

In [ ]:
# Example: Search for a feature layer item in your AGOL content and add it to the map widget
# (Uncomment and replace with your actual AGOL Item ID if you published one earlier!)

# my_item = gis.content.get("YOUR_ITEM_ID_HERE")
# if my_item:
#     m_arcgis.add_layer(my_item)
#     print(f"Added layer '{my_item.title}' to the ArcGIS map widget!")
# else:
#     print("Item ID not found. You can also drag and drop layers from the Pro catalog directly into your project.")